## Harmony Py Library
### Job Steps Example



In [ ]:
import json
from harmony import BBox, Client, Collection, Request, Environment, StepsRequest

import helper
helper.install_project_and_dependencies('..')

In [ ]:
harmony_client = Client(env=Environment.UAT)  # assumes .netrc usage
request = Request(
    collection=Collection(id='C1268429309-EEDTEST'),
    granule_id=['G1281797307-EEDTEST'],
    format='image/tiff',
    variables=['Soil_Moisture_Retrieval_Data/landcover_class_fraction'],
    labels=['harmony-py-steps-example'],
)



In [ ]:
# submit an async request for processing and return the job_id
job_id = harmony_client.submit(request)
job_id

harmony_client.wait_for_processing(job_id)


Form a simple request to the steps endpoint

In [ ]:

step_request = StepsRequest(
    job_id=job_id, 
)
response = harmony_client.submit(step_request)
print(json.dumps(response, indent=2))

Show the input and output files for the smap-l2-gridder step.


We need to find the work_item of the harmony-smap-l2-gridder step to filter the resolve files results.

In [ ]:
workitem_id = next(
    step["workItems"][0]["id"]
    for step in response["steps"]
    if "smap-l2-gridder" in step["serviceID"]
)
workitem_id

Make the request to resolve the input and output files.

In [ ]:
step_files_request = StepsRequest(
    job_id=job_id, 
    work_item=[workitem_id],
    resolve_files=True, 
)
response = harmony_client.submit(step_files_request)
print(json.dumps(response, indent=2))

In [ ]:
input_file = response["steps"][0]["workItems"][0]["inputFiles"][0]

output_file = response["steps"][0]["workItems"][0]["outputFiles"][0]

In [ ]:
!curl -n -L -o "Soil_Moisture_Retrieval_Data_landcover_class_fraction_subsetted.h5" "{input_file}"

!curl -n -L -o "Soil_Moisture_Retrieval_Data_landcover_class_fraction_subsetted_regridded.nc" "{output_file}"